
# 第 2 章 神经网络的数学基础

目标：
1. 认识神经网络
2. 掌握张量与张量预算
3. 理解并掌握神经网络如何通过反向传播与梯度下降进行学习

## 2.1 初始神经网络



In [22]:
# 代码清单2-1：使用Keras加载 MNIST数据集
from tensorflow.keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print(train_images.shape)
print(train_labels.shape)
print(test_images.shape)
print(test_labels.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


我们的工作流程如下：首先，将训练数据（train_images和train_labels）输入神经网络；然后，神经网络学习将图像和标签关联在一起；最后，神经网络对test_images进行预测，我们来验证这些预测与test_labels中的标签是否匹配。

下面我们来构建神经网络。

In [23]:
# 代码清单2-2：神经网络架构
from tensorflow import keras
from tensorflow.keras import models

model = keras.Sequential([
    layers.Dense(512, activation='relu'),
    layers.Dense(10, activation='softmax')
])

神经网络的核心组件是层（layer）​。你可以将层看成数据过滤器：进去一些数据，出来的数据变得更加有用。具体来说，层从输入数据中提取表示——我们期望这种表示有助于解决手头的问题。大多数深度学习工作涉及将简单的层链接起来，从而实现渐进式的数据蒸馏（data distillation）​。深度学习模型就像是处理数据的筛子，包含一系列越来越精细的数据过滤器（也就是层）​。

本例中的模型包含2个Dense层，它们都是密集连接（也叫全连接）的神经层。第2层（也是最后一层）是一个10路softmax分类层，它将返回一个由10个概率值（总和为1）组成的数组。每个概率值表示当前数字图像属于10个数字类别中某一个的概率。

在训练模型之前，我们还需要指定编译（compilation）步骤的3个参数。

优化器（optimizer）​：模型基于训练数据来自我更新的机制，其目的是提高模型性能。损失函数（loss function）​：模型如何衡量在训练数据上的性能，从而引导自己朝着正确的方向前进。在训练和测试过程中需要监控的指标（metric）​：本例只关心精度（accuracy）​，即正确分类的图像所占比例。

本章我们先建立映像，后面会深入学习。

In [24]:
# 代码清单2-3：编译步骤
model.compile(
    optimizer='rmsprop',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

在开始训练之前，我们先对数据进行预处理，将其变换为模型要求的形状，并缩放到所有值都在[0, 1]区间。前面提到过，训练图像保存在一个uint8类型的数组中，其形状为(60000, 28, 28)，取值区间为[0, 255]。我们将把它变换为一个float32数组，其形状为(60000, 28 * 28)，取值范围是[0, 1]。下面准备图像数据：

In [25]:
# 代码清单2-4：图像数据处理
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype('float32') / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype('float32') / 255

现在我们准备开始训练模型。在Keras中，这一步是通过调用模型的fit方法来完成的——我们在训练数据上拟合（fit）模型：

In [26]:
# 代码清单2-5：拟合模型
model.fit(train_images, train_labels, epochs=5, batch_size=128)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9250 - loss: 0.2625
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9678 - loss: 0.1080
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9787 - loss: 0.0711
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9844 - loss: 0.0514
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9887 - loss: 0.0388


训练过程中显示了两个数字：一个是模型在训练数据上的损失值（loss）​，另一个是模型在训练数据上的精度（acc）​。我们很快就在训练数据上达到了0.989（98.9%）的精度。

现在我们得到了一个训练好的模型，可以利用它来预测新数字图像的类别概率​。这些新数字图像不属于训练数据，比如可以是测试集中的数据。

In [31]:
# 代码清单2-6：利用模型进行预测
test_digits = test_images[:10]
predictions = model.predict(test_digits)
print(predictions[0])
print(predictions[0].argmax() == test_labels[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
[3.2180605e-08 3.7997996e-09 4.8086940e-06 1.4444927e-04 2.1575432e-11
 3.6309796e-08 3.1179085e-12 9.9984896e-01 1.9740834e-07 1.3952014e-06]
True


我们最后使用全部的测试数据进行模型的精度计算：

In [32]:
# 代码清单2-7：在测试数据上评估模型
model.evaluate(test_images, test_labels)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9778 - loss: 0.0659


[0.06586765497922897, 0.9778000116348267]

测试精度约为97.8%，比训练精度（98.9%）低不少。训练精度和测试精度之间的这种差距是过拟合（overfit）造成的。过拟合是指机器学习模型在新数据上的性能往往比在训练数据上要差。后续章节我们想学习防止过拟合的优化手段。

## 2.2 神经网络的数据表示

在前面的例子中，我们的数据存储在多维NumPy数组中，也叫作张量（tensor）​。一般来说，目前所有机器学习系统都使用张量作为基本数据结构。

### 2.2.1 标量（0阶张量）


In [33]:
import numpy as np

x = np.array(12)
print(x)
print(x.ndim)

12
0



### 2.2.2 向量（1阶张量）

In [34]:
x = np.array([12, 3, 6, 14, 7])
print(x)
print(x.ndim)

[12  3  6 14  7]
1



### 2.2.3 矩阵（2阶张量）

In [35]:
x = np.array([
    [5, 78, 2, 34, 0],
    [6, 79, 3, 35, 1],
    [7, 80, 4, 36, 2]
])
print(x)
print(x.ndim)

[[ 5 78  2 34  0]
 [ 6 79  3 35  1]
 [ 7 80  4 36  2]]
2


### 2.2.4 3阶张量（更高阶张量）

将多个矩阵打包成一个新的数组，就可以得到一个3阶张量。我们可以将其只管地理解为数字组成的立方体。

### 2.2.5 关键属性

